In [1]:
# Cell 1 — Configuration

from pathlib import Path
from PIL import Image, ImageOps
import pandas as pd
import json, re, shutil

EXPORTS_ROOT = Path("/Users/jwatts/Documents/astrophotography/Exports")
DSO_JSON     = Path("mobile/dso.json")  # existing catalog with metadata
OUTPUT_DIR   = Path("web")
JPEG_QUALITY = 95
MAX_FILE_SIZE = 10 * 1024 * 1024  # 10 MB Cloudinary free tier limit
MIN_QUALITY  = 80                  # never go below this
IMAGE_EXTENSIONS = {".jpg", ".jpeg"}

# Cloudinary config
CLOUDINARY_ROOT = "island-skies-astro"
CLOUDINARY_FOLDER_MAP = {
    "dso":          "dso",
    "solar-system": "solarsystem",
}

# Map Exports folders → website categories
CATEGORY_MAP = {
    "DSO":    "dso",
    "Planet": "solar-system",
    "Comet":  "solar-system",
    "Lunar":  "solar-system",
}

print(f"Exports root: {EXPORTS_ROOT}")
print(f"Output dir:   {OUTPUT_DIR.resolve()}")

Exports root: /Users/jwatts/Documents/astrophotography/Exports
Output dir:   /Users/jwatts/StudioProjects/git/AstroPlannerData/web


In [2]:
# Cell 2 — Helper functions

def to_kebab(s: str) -> str:
    """CamelCase/underscores/hyphens → kebab-case."""
    s = re.sub(r'(?<=[a-z0-9])(?=[A-Z])', '-', s)
    s = s.replace('_', '-')
    return s.lower()


def make_image_id(object_id: str, region: str | None, filename_stem: str) -> str:
    """Generate a unique, URL-friendly image ID."""
    stem = to_kebab(filename_stem)
    if region:
        return f"{object_id}-{to_kebab(region)}"
    if stem == object_id or stem.replace('-', '') == object_id:
        return object_id
    if stem.startswith(object_id):
        return stem
    return f"{object_id}-{stem}"


def format_catalog_id(object_id: str) -> str:
    """ic443 → IC 443, m42 → M 42, ngc7000 → NGC 7000, barnard68 → Barnard 68."""
    match = re.match(r'^([a-zA-Z]+)(\d+)$', object_id)
    if match:
        prefix, number = match.groups()
        if prefix.lower() in ('m', 'ngc', 'ic'):
            return f"{prefix.upper()} {number}"
        return f"{prefix.capitalize()} {number}"
    return object_id


# Quick sanity checks
assert to_kebab("PillarsOfCreation") == "pillars-of-creation"
assert to_kebab("IC1805Inner") == "ic1805-inner"
assert to_kebab("jupiter_grs_io") == "jupiter-grs-io"
assert make_image_id("m16", "PillarsOfCreation", "pillars") == "m16-pillars-of-creation"
assert make_image_id("ic434", None, "IC434") == "ic434"
assert make_image_id("ic434", None, "IC4342") == "ic4342"
assert make_image_id("ic410", None, "tadpoles") == "ic410-tadpoles"
assert make_image_id("m42", None, "m42-2") == "m42-2"
assert format_catalog_id("ic443") == "IC 443"
assert format_catalog_id("ngc7000") == "NGC 7000"
assert format_catalog_id("barnard68") == "Barnard 68"
print("All checks passed.")

All checks passed.


In [3]:
# Cell 3 — Discovery functions

def discover_dso() -> list[dict]:
    """Walk DSO/ recursively for all .jpg files.
    Direct images: DSO/<Object>/<file>.jpg
    Region images: DSO/<Object>/<Region>/<file>.jpg
    """
    results = []
    dso_root = EXPORTS_ROOT / "DSO"
    if not dso_root.exists():
        print(f"WARNING: {dso_root} not found")
        return results

    for jpg in sorted(dso_root.rglob("*")):
        if not jpg.is_file() or jpg.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        rel = jpg.relative_to(dso_root)
        parts = rel.parts

        object_id = parts[0].lower()

        if len(parts) == 2:
            region = None
        elif len(parts) == 3:
            region = parts[1]
        else:
            print(f"  WARNING: unexpected depth {rel}")
            continue

        image_id = make_image_id(object_id, region, jpg.stem)

        results.append({
            "imageId": image_id,
            "objectId": object_id,
            "region": region,
            "category": "dso",
            "src_path": str(jpg),
            "filename": jpg.name,
        })

    return results


def discover_planets() -> list[dict]:
    """Walk Planet/ — session subfolders become separate images.
    Direct images: Planet/<Planet>/<file>.jpg → imageId = planet name
    Session folders: Planet/<Planet>/<session>/<file>.jpg → imageId = session name
    """
    results = []
    planet_root = EXPORTS_ROOT / "Planet"
    if not planet_root.exists():
        print(f"WARNING: {planet_root} not found")
        return results

    for planet_folder in sorted(planet_root.iterdir()):
        if not planet_folder.is_dir():
            continue
        planet_name = planet_folder.name.lower()

        # Direct images in planet folder
        direct_images = sorted(
            f for f in planet_folder.iterdir()
            if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
        )
        for i, img in enumerate(direct_images):
            img_id = planet_name if len(direct_images) == 1 else f"{planet_name}-{i+1}"
            results.append({
                "imageId": img_id,
                "objectId": planet_name,
                "region": None,
                "category": "solar-system",
                "src_path": str(img),
                "filename": img.name,
            })

        # Session subfolders
        for session_folder in sorted(planet_folder.iterdir()):
            if not session_folder.is_dir():
                continue
            session_name = to_kebab(session_folder.name)
            img = next(
                (f for f in sorted(session_folder.iterdir())
                 if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS),
                None,
            )
            if img:
                results.append({
                    "imageId": session_name,
                    "objectId": planet_name,
                    "region": None,
                    "category": "solar-system",
                    "src_path": str(img),
                    "filename": img.name,
                })

    return results


def discover_lunar() -> list[dict]:
    """Walk Lunar/<body>/<feature>/ — each feature subfolder is one image."""
    results = []
    lunar_root = EXPORTS_ROOT / "Lunar"
    if not lunar_root.exists():
        print(f"WARNING: {lunar_root} not found")
        return results

    for body_folder in sorted(lunar_root.iterdir()):
        if not body_folder.is_dir():
            continue
        body_name = body_folder.name.lower()

        for feature_folder in sorted(body_folder.iterdir()):
            if not feature_folder.is_dir():
                continue
            feature_name = feature_folder.name.lower()
            img = next(
                (f for f in sorted(feature_folder.iterdir())
                 if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS),
                None,
            )
            if img:
                results.append({
                    "imageId": f"{body_name}-{feature_name}",
                    "objectId": body_name,
                    "region": feature_name,
                    "category": "solar-system",
                    "src_path": str(img),
                    "filename": img.name,
                })

    return results


def discover_comets() -> list[dict]:
    """Walk Comet/<name>/ — one image per comet folder."""
    results = []
    comet_root = EXPORTS_ROOT / "Comet"
    if not comet_root.exists():
        print(f"WARNING: {comet_root} not found")
        return results

    for comet_folder in sorted(comet_root.iterdir()):
        if not comet_folder.is_dir():
            continue
        comet_name = comet_folder.name.lower()
        img = next(
            (f for f in sorted(comet_folder.iterdir())
             if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS),
            None,
        )
        if img:
            results.append({
                "imageId": comet_name,
                "objectId": comet_name,
                "region": None,
                "category": "solar-system",
                "src_path": str(img),
                "filename": img.name,
            })

    return results


print("Discovery functions defined.")

Discovery functions defined.


In [4]:
# Cell 4 — Run discovery, build DataFrame, join with DSO catalog

all_images = []
all_images.extend(discover_dso())
all_images.extend(discover_planets())
all_images.extend(discover_lunar())
all_images.extend(discover_comets())

images_df = pd.DataFrame(all_images)

# Load DSO catalog for metadata
dso_catalog = pd.read_json(DSO_JSON)
dso_meta = dso_catalog[["objectId", "displayName", "ra", "dec", "constellation", "type", "subType"]].copy()

# Left join to get DSO metadata
images_df = images_df.merge(dso_meta, on="objectId", how="left")

# Generate display names
def make_display_name(row):
    base_name = row.get("displayName")
    if pd.notna(base_name):
        if pd.notna(row.get("region")) and row["category"] == "dso":
            # Add human-readable region name for DSO region images
            region_pretty = re.sub(r'(?<=[a-z0-9])(?=[A-Z])', ' ', str(row["region"]))
            return f"{base_name} - {region_pretty}"
        return base_name
    # Fallback: capitalize objectId
    return row["objectId"].replace("-", " ").title()

images_df["displayName"] = images_df.apply(make_display_name, axis=1)

# Print summary
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 60)
print(f"{'imageId':<35} {'category':<15} {'objectId':<12} {'displayName'}")
print("-" * 110)
for _, row in images_df.iterrows():
    print(f"  {row['imageId']:<33} {row['category']:<15} {row['objectId']:<12} {row['displayName']}")

dso_count = len(images_df[images_df["category"] == "dso"])
ss_count = len(images_df[images_df["category"] == "solar-system"])
print(f"\nTotal: {len(images_df)} images ({dso_count} DSO, {ss_count} solar-system)")

# Check for duplicate imageIds
dupes = images_df[images_df.duplicated(subset="imageId", keep=False)]
if len(dupes) > 0:
    print(f"\nWARNING: {len(dupes)} duplicate imageIds:")
    print(dupes[["imageId", "src_path"]])

imageId                             category        objectId     displayName
--------------------------------------------------------------------------------------------------------------
  ic1805                            dso             ic1805       Heart Nebula
  ic1805-ic1805-inner               dso             ic1805       Heart Nebula - IC1805 Inner
  ic410                             dso             ic410        Tadpoles Nebula
  ic410-tadpoles                    dso             ic410        Tadpoles Nebula
  ic434                             dso             ic434        Horsehead Nebula
  ic4342                            dso             ic434        Horsehead Nebula
  ic4343                            dso             ic434        Horsehead Nebula
  ic443                             dso             ic443        Jellyfish Nebula
  ic5146                            dso             ic5146       Cocoon Nebula
  m1                                dso             m1           Crab Ne

In [5]:
# Cell 5 — Process images (DSO: JPEG re-compression; solar-system: copy as-is)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "dso").mkdir(exist_ok=True)
(OUTPUT_DIR / "solar-system").mkdir(exist_ok=True)

widths = []
heights = []

for _, row in images_df.iterrows():
    src = Path(row["src_path"])
    dst = OUTPUT_DIR / row["category"] / f"{row['imageId']}.jpg"

    with Image.open(src) as img:
        img = ImageOps.exif_transpose(img)
        w, h = img.size
        widths.append(w)
        heights.append(h)

        if row["category"] == "solar-system":
            # Planetary/lunar/comet images are small — copy original unchanged
            shutil.copy2(src, dst)
            size_mb = dst.stat().st_size / (1024 * 1024)
            print(f"  {row['imageId']:<35} {w}x{h}  →  {size_mb:.1f} MB (copied)")
        else:
            # DSO images: re-compress, step down quality if over 10 MB
            quality = JPEG_QUALITY
            while quality >= MIN_QUALITY:
                img.save(dst, format="JPEG", quality=quality, optimize=True)
                file_size = dst.stat().st_size
                if file_size <= MAX_FILE_SIZE:
                    break
                quality -= 2

            size_mb = file_size / (1024 * 1024)
            flag = " ⚠️ OVER 10MB" if file_size > MAX_FILE_SIZE else ""
            q_note = f" (q={quality})" if quality != JPEG_QUALITY else ""
            print(f"  {row['imageId']:<35} {w}x{h}  →  {size_mb:.1f} MB{q_note}{flag}")

images_df["width"] = widths
images_df["height"] = heights

print(f"\nDone. {len(images_df)} images written to {OUTPUT_DIR.resolve()}")

  ic1805                              6240x4162  →  8.5 MB
  ic1805-ic1805-inner                 4144x2822  →  1.6 MB
  ic410                               4052x2674  →  1.6 MB
  ic410-tadpoles                      4144x2822  →  1.6 MB
  ic434                               4100x2775  →  1.7 MB
  ic4342                              2800x2800  →  0.9 MB
  ic4343                              7356x2780  →  2.9 MB
  ic443                               6238x4167  →  8.2 MB
  ic5146                              4136x2811  →  2.8 MB
  m1                                  3907x2585  →  1.7 MB
  m101                                4129x2811  →  1.6 MB
  m13                                 4138x2820  →  6.3 MB
  m16-pillars-of-creation             4144x2822  →  1.1 MB
  m16                                 4134x2815  →  2.2 MB
  m20                                 4118x2791  →  2.4 MB
  m27                                 4126x2814  →  3.1 MB
  m3                                  4134x2818  →  2.1 

In [6]:
# Cell 6 — Generate manifest (web/images.json)

def make_alt_text(row):
    """Generate descriptive alt text for accessibility."""
    if row["category"] == "dso":
        catalog = format_catalog_id(row["objectId"])
        name = row["displayName"]
        return f"{catalog} - {name}" if name != catalog else catalog
    return row["displayName"]


def make_cloudinary_id(row):
    folder = CLOUDINARY_FOLDER_MAP[row["category"]]
    return f"{CLOUDINARY_ROOT}/{folder}/{row['imageId']}"


manifest = []
for _, row in images_df.iterrows():
    entry = {
        "id": row["imageId"],
        "title": row["displayName"],
        "category": row["category"],
        "objectId": row["objectId"],
        "region": row["region"] if pd.notna(row.get("region")) else None,
        "cloudinaryId": make_cloudinary_id(row),
        "altText": make_alt_text(row),
        "objectName": row["displayName"],
        "catalogId": format_catalog_id(row["objectId"]) if row["category"] == "dso" else None,
        "ra": row["ra"] if pd.notna(row.get("ra")) else None,
        "dec": row["dec"] if pd.notna(row.get("dec")) else None,
        "constellation": row["constellation"] if pd.notna(row.get("constellation")) else None,
        "objectType": row["type"] if pd.notna(row.get("type")) else None,
        "objectSubType": row["subType"] if pd.notna(row.get("subType")) else None,
        "filename": f"{row['imageId']}.jpg",
        "width": int(row["width"]),
        "height": int(row["height"]),
    }
    manifest.append(entry)

manifest_path = OUTPUT_DIR / "images.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"Wrote {len(manifest)} entries to {manifest_path.resolve()}")

# Show first 3 entries as sample
print("\nSample entries:")
print(json.dumps(manifest[:3], indent=2))

Wrote 55 entries to /Users/jwatts/StudioProjects/git/AstroPlannerData/web/images.json

Sample entries:
[
  {
    "id": "ic1805",
    "title": "Heart Nebula",
    "category": "dso",
    "objectId": "ic1805",
    "region": null,
    "cloudinaryId": "island-skies-astro/dso/ic1805",
    "altText": "IC 1805 - Heart Nebula",
    "objectName": "Heart Nebula",
    "catalogId": "IC 1805",
    "ra": 2.5448639,
    "dec": 61.4568889,
    "constellation": "Cassiopeia",
    "objectType": "nebula",
    "objectSubType": "Emission Nebula",
    "filename": "ic1805.jpg",
    "width": 6240,
    "height": 4162
  },
  {
    "id": "ic1805-ic1805-inner",
    "title": "Heart Nebula - IC1805 Inner",
    "category": "dso",
    "objectId": "ic1805",
    "region": "IC1805Inner",
    "cloudinaryId": "island-skies-astro/dso/ic1805-ic1805-inner",
    "altText": "IC 1805 - Heart Nebula - IC1805 Inner",
    "objectName": "Heart Nebula - IC1805 Inner",
    "catalogId": "IC 1805",
    "ra": 2.5448639,
    "dec": 61.4568

In [ ]:
# Cell 7 — Cloudinary upload

import os
from dotenv import load_dotenv
import cloudinary
import cloudinary.uploader

load_dotenv()

cloudinary.config(
    cloud_name=os.environ["CLOUDINARY_CLOUD_NAME"],
    api_key=os.environ["CLOUDINARY_API_KEY"],
    api_secret=os.environ["CLOUDINARY_API_SECRET"],
)

for _, row in images_df.iterrows():
    path = OUTPUT_DIR / row["category"] / f"{row['imageId']}.jpg"
    public_id = make_cloudinary_id(row)
    result = cloudinary.uploader.upload(
        str(path),
        public_id=public_id,
        overwrite=True,
        resource_type="image",
    )
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"  {row['imageId']:<35} {size_mb:.1f} MB  →  {result['secure_url']}")

print(f"\nUploaded {len(images_df)} images to Cloudinary.")